In [ ]:
import random
random.seed(42)
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, pairwise_distances
import pandas as pd
import numpy as np
import time
np.random.seed(42)
torch.manual_seed(42)
import data_preprocessor
import utils
import pickle
from sklearn import preprocessing, model_selection, metrics
from pyclassifiers import ImprovedSimpleFNN,ResNetClassifier
from pytorch_models import SimplePytorchModel,PytorchModel,ModelEvaluator
import usi_data_selection
import simulate_missingness
import data_preprocessor


In [ ]:
# Main for Data preparation
exp_no = 1
trial_no = 1
this_dataset_name = 'income'
new_directory = f'{this_dataset_name}_exp{exp_no}'
# Check if the directory already exists
if not os.path.exists(new_directory):
    # Create the directory
    os.makedirs(new_directory)
    print(f"Directory '{new_directory}' was created successfully.")
else:
    print(f"Directory '{new_directory}' already exists.")

# Create a subdirectory for the results
this_dirt_path = os.path.join(os.getcwd(), new_directory)

this_trial_path = os.path.join(this_dirt_path, f'trial_{trial_no}')
if not os.path.exists(this_trial_path):
    os.makedirs(this_trial_path)
    print(f"Subdirectory '{this_trial_path}' was created successfully.")
else:
    print(f"Subdirectory '{this_trial_path}' already exists.")

this_model_name = 'RESNET'
this_model_path = os.path.join(this_trial_path, this_model_name)
if not os.path.exists(this_model_path):
    os.makedirs(this_model_path)
    print(f"Subdirectory '{this_model_path}' was created successfully.")
else:
    print(f"Subdirectory '{this_model_path}' already exists.")

In [ ]:
# Training dataset for building the model
# Development dataset for evaluating the Shapley value of each training data point
# Validation dataset for evaluating the model performance during training process
# Testing dataset for evaluating the model performance
this_data_path = f'dataset/{this_dataset_name}'
if not os.path.exists(this_data_path):
    os.makedirs(this_data_path)
    print(f"Directory '{this_data_path}' was created successfully.")
else:
    print(f"Directory '{this_data_path}' already exists.")

if this_dataset_name == 'income':
    data = pd.read_csv('dataset/income/raw_data.csv',index_col=0)
    X = data.drop(columns='income')
    y = data['income']

label_encoder = preprocessing.LabelEncoder()
label_encoder.fit(y)
encoded = label_encoder.transform(y)
target_y = pd.DataFrame(encoded, columns=['target'],index=y.index)


train_testX, val_devX, train_testY, val_devY = model_selection.train_test_split(X, target_y, test_size=0.2,stratify=target_y,random_state=42)

# For classification, we can stratify all splits
trainX, testX, trainY, testY = model_selection.train_test_split(train_testX, train_testY, test_size=0.3, stratify=train_testY, random_state=42)

validationX, developX, validationY, developY = model_selection.train_test_split(val_devX, val_devY, test_size=0.1, stratify=val_devY, random_state=42)

print(f'Training+ testing data: {train_testX.shape[0]}, Validation+Development data: {val_devX.shape[0]}')
print(f'Training data: {trainX.shape[0]}, Developing data: {developX.shape[0]}')
print(f'Validation data: {validationX.shape[0]}, Testing data: {testX.shape[0]}')

trainX.sort_index(ascending=True, inplace=True)
trainY.sort_index(ascending=True,inplace=True)
developX.sort_index(ascending=True, inplace=True)
developY.sort_index(ascending=True,inplace=True)
validationX.sort_index(ascending=True, inplace=True)
validationY.sort_index(ascending=True,inplace=True)
testX.sort_index(ascending=True,inplace=True)
testY.sort_index(ascending=True,inplace=True)
print(trainX.index)
trainX.index.name= 'train_idx'
trainY.index.name= 'train_idx'
train = pd.concat([trainX,trainY],axis=1)
train.to_csv(f'{this_data_path}/train.csv',index=True)
test = pd.concat([testX,testY],axis=1)
test.to_csv(f'{this_data_path}/test.csv',index=True)
develop = pd.concat([developX,developY],axis=1)
develop.to_csv(f'{this_data_path}/develop.csv',index=True)
validation = pd.concat([validationX,validationY],axis=1)
validation.to_csv(f'{this_data_path}/validation.csv',index=True)
this_raw_train_features = trainX.columns
print(this_raw_train_features)
trnX_idx = train.index.tolist()
this_data_path = f'dataset/{this_dataset_name}'
train = pd.read_csv(f'{this_data_path}/train.csv',index_col='train_idx')
test = pd.read_csv(f'{this_data_path}/test.csv',index_col=0)
develop = pd.read_csv(f'{this_data_path}/develop.csv',index_col=0)
validation = pd.read_csv(f'{this_data_path}/validation.csv',index_col=0)
trnX_idx = train.index.tolist()
if this_dataset_name == 'income':
    target_name = 'target'
    categorical_columns = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
    numerical_columns = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week','fnlwgt']
   


#### Evaluate Fully-Observed dataset as upperbound

In [ ]:
# Load the original dataset and evaluate the original dataset

evaluate_original_benchmark = True # False: read from pkl file
if evaluate_original_benchmark == True:
    performance_file_name = None    
    data_variant_file_name = None
else:
    performance_file_name = 'income_exp1\\trial_1\\RESNET\\MNAR_obs_feature_frac_0.3_benchmark_performance_dict.pkl'
    data_variant_file_name = 'income_exp1\\trial_1\\RESNET\\MNAR_obs_feature_frac_0.3_this_all_dataset_variants.pkl'

this_data_path = f'dataset/{this_dataset_name}'
# original_performance_threshold = 0.819
og_train = pd.read_csv(f'{this_data_path}/train.csv',index_col='train_idx')
og_test = pd.read_csv(f'{this_data_path}/test.csv',index_col=0)
og_develop = pd.read_csv(f'{this_data_path}/develop.csv',index_col=0)
og_validation = pd.read_csv(f'{this_data_path}/validation.csv',index_col=0)

og_trainX = og_train.drop(columns=[target_name])
og_developX = og_develop.drop(columns=[target_name])
og_validationX = og_validation.drop(columns=[target_name])
og_testX = og_test.drop(columns=[target_name])

og_trnY = og_train[target_name].values
og_devY = og_develop[target_name].values
og_valY = og_validation[target_name].values
og_tstY = og_test[target_name].values


# Read or Evaluate the original dataset performance
if evaluate_original_benchmark == True:
    # Create original dataset variants
    this_all_dataset_variants = {}
    og_encoded_trnX, og_encoded_valX,  og_encoded_devX, og_encoded_tstX, og_feature_names = data_preprocessor.DataPreprocessor.encode_categorical_scale_numerical(og_trainX,og_validationX,og_developX,og_testX,categorical_columns,numerical_columns)
    this_all_dataset_variants['original'] = {
        'trnX': og_encoded_trnX,
        'trnY': og_trnY,
        'valX': og_encoded_valX,
        'valY': og_valY,
        'devX': og_encoded_devX,
        'devY': og_devY,
        'tstX': og_encoded_tstX,
        'tstY': og_tstY,
        'feature_names': og_feature_names,
    }
    print(f'Original dataset shape:{og_encoded_trnX.shape}')
    this_benchmark_performance_dict = {
        'original': {},
    }
    # Original: original dataset is the uppper bound
    print('Evaluating Original Dataset ...')
    original_trnX = this_all_dataset_variants['original']['trnX']
    original_trnY = this_all_dataset_variants['original']['trnY']
    original_tstX = this_all_dataset_variants['original']['tstX']
    original_tstY = this_all_dataset_variants['original']['tstY']
    original_valX = this_all_dataset_variants['original']['valX']
    original_valY = this_all_dataset_variants['original']['valY']
    original_model_config = {
        'model_name': 'RESNET',
        'trnX': original_trnX,
        'trnY': original_trnY,
        'tstX': original_tstX,
        'tstY': original_tstY,
        'valX': original_valX,
        'valY': original_valY,
        'epochs': 100,
        'eval_metrics': 'both',
        'hidden_layer_list': [256,128,64],
        'set_scheduler': True,
    }
    original_score_dict, original_proba, original_model_params, original_training_time = ModelEvaluator(original_model_config).evaluate_avg_metrics(total_count=1)
    this_benchmark_performance_dict['original']['time'] = original_training_time
    this_benchmark_performance_dict['original']['score'] = original_score_dict
else:
    this_benchmark_performance_dict = {
        'original': {},
    }
    this_all_dataset_variants = {
        'original': {},
    }
    original_score_dict = utils.read_dict_from_pkl(performance_file_name)['original']['score']
    
    this_benchmark_performance_dict['original']['score'] = original_score_dict
    print(this_benchmark_performance_dict["original"])
    original_data_variant = utils.read_dict_from_pkl(data_variant_file_name)['original']
    this_all_dataset_variants['original'] = original_data_variant


print(this_benchmark_performance_dict)
print(this_all_dataset_variants)

#### Simulate missingness (MCAR/MAR/MNAR)

In [ ]:
# Simulation settings (row-incomplete rate + mechanism + beta heterogeneity)
simulate_new = True

this_total_missing_rows_rate = 0.9         # controls % rows that have >=1 missing (row gate)
this_cell_missing_rate = 0.3               # controls % cells that become missing (column gate)
this_missing_mechanism = "MCAR"             # "MCAR" | "MAR" | "MNAR"

# "always observed" feature fraction (for MAR/MNAR)
this_observed_feature_frac = 0.3

# Beta column heterogeneity (for maskable columns only)
this_beta_mean = 0.3                        # mean of Beta base-rate across maskable columns
this_beta_concentration = 5.0               # higher => less heterogeneity

# Logistic MAR strength knobs (optional; keep defaults if you don't want to tune)
this_w_scale = 0.5
this_global_logit_shift = 0.0

# MNAR self-masking strength (only used if this_missing_mechanism == "MNAR")
this_mnar_lambda_scale = 1.0

# Record settings for reproducibility
this_simulate_setting_dict = {
    "missing_row_rate": this_total_missing_rows_rate,
    "mechanism": this_missing_mechanism,
    "observed_feature_frac": 0 if this_missing_mechanism == "MCAR" else this_observed_feature_frac,
    "beta_mean": this_beta_mean,
    "beta_concentration": this_beta_concentration,
    "w_scale": 0 if this_missing_mechanism == "MCAR" else this_w_scale,
    "mnar_lambda_scale": 0 if this_missing_mechanism == "MCAR" else this_mnar_lambda_scale,
}

this_file_suffix = f'{this_simulate_setting_dict.get("mechanism")}_obs_feature_frac_{this_simulate_setting_dict.get("observed_feature_frac")}'
if not os.path.exists(f"{this_dataset_name}_exp{exp_no}/data_{this_file_suffix}"):
    os.makedirs(f"{this_dataset_name}_exp{exp_no}/data_{this_file_suffix}")

utils.write_dict_to_pkl(
    this_simulate_setting_dict,
    f"{this_dataset_name}_exp{exp_no}/data_{this_file_suffix}/this_simulate_setting_dict.pkl"
)
print(f"Simulate New? {simulate_new}, {this_simulate_setting_dict}")

if simulate_new:
    for cleaned_df, split_name in [
        (train, "train"),
        (develop, "develop"),
        (validation, "validation"),
        (test, "test"),
    ]:
        missed_df, diag = simulate_missingness.simulate_missing_df(
            cleaned_df=cleaned_df,
            target_col=target_name,
            mechanism=this_missing_mechanism,
            missing_row_rate=this_total_missing_rows_rate,
            observed_feature_frac=this_observed_feature_frac,
            beta_mean=this_beta_mean,
            beta_concentration=this_beta_concentration,
            w_scale=this_w_scale,
            mnar_lambda_scale=this_mnar_lambda_scale,
            random_state=42,
        )

        missed_df.to_csv(f"{this_dataset_name}_exp{exp_no}/data_{this_file_suffix}/{split_name}_incomplete.csv")

        print(split_name)
        print("Total rows:", missed_df.shape[0])
        print("Complete rows:", missed_df.notna().all(axis=1).sum())
        print("Missing rows:", missed_df.isna().any(axis=1).sum())
        print(missed_df[missed_df.isna().any(axis=1)].head(2))
        print()

# Write per-column missingness actually achieved (from diag)
with open(f"{this_dataset_name}_exp{exp_no}/data_{this_file_suffix}/col_missingness_distribution.txt", "w") as f:
    f.write("Column-level missingness (fraction of NaNs per column):\n")
    for col, frac in diag["achieved_col_missingness"].items():
        f.write(f"{col}: {frac:.3f}\n")

#### Evaluate Drop-All incomplete rows as lower-bound.

In [ ]:
# ---- CONFIG ----
target_name = "target"

# Load simulated incomplete splits
simulate_train_df = pd.read_csv(f"{this_dataset_name}_exp{exp_no}/data_{this_file_suffix}/train_incomplete.csv", index_col=0)
simulate_develop_df = pd.read_csv(f"{this_dataset_name}_exp{exp_no}/data_{this_file_suffix}/develop_incomplete.csv", index_col=0)
simulate_test_df = pd.read_csv(f"{this_dataset_name}_exp{exp_no}/data_{this_file_suffix}/test_incomplete.csv", index_col=0)
simulate_validation_df = pd.read_csv(f"{this_dataset_name}_exp{exp_no}/data_{this_file_suffix}/validation_incomplete.csv", index_col=0)


# create_variants == True, create complete_samples, incomplete_with_indicators, per_feature_marginals
dataprocess_config_create_variants = {
    "train_df": simulate_train_df,
    "val_df": og_validation,
    "dev_df": simulate_develop_df,
    "test_df": og_test,
    "target_col": target_name,
    "categorical_cols": categorical_columns,
    "numerical_cols": numerical_columns,
    # complete_samples is created when create_variant=True; imputation_method irrelevant for complete_samples itself
    "imputation_method": {"numerical": "mean", "categorical": "__MISSING__"},
    "create_variant": True,
    "process_incomplete_val_test": False,
}
variants_complete_samples_etc = data_preprocessor.DataPreprocessor(dataprocess_config_create_variants).preprocess()
this_all_dataset_variants["complete_samples"] = variants_complete_samples_etc["complete_samples"]
this_all_dataset_variants["incomplete_with_indicators"] = variants_complete_samples_etc["incomplete_with_indicators"]
this_all_dataset_variants["per_feature_marginals"] = variants_complete_samples_etc["per_feature_marginals"]
print(this_all_dataset_variants.keys())
utils.write_dict_to_pkl(this_all_dataset_variants, f"{this_model_path}/{this_file_suffix}_this_all_dataset_variants.pkl")

this_benchmark_performance_dict['complete_samples'] = {}


print("Evaluating complete_samples ...")
tmp = this_all_dataset_variants["complete_samples"]
complete_samples_model_config = {
    "model_name": "RESNET",
    "trnX": tmp["trnX"], "trnY": tmp["trnY"],
    "tstX": tmp["tstX"], "tstY": tmp["tstY"],
    "valX": tmp["valX"], "valY": tmp["valY"],
    "epochs": 100,
    "eval_metrics": "both",
    "hidden_layer_list": [256, 128, 64],
    "set_scheduler": True,
}
score_dict, proba, model_params, training_time = ModelEvaluator(complete_samples_model_config).evaluate_avg_metrics(total_count=1)
this_benchmark_performance_dict["complete_samples"] = {"time": training_time, "score": score_dict}
print(f"complete_samples roc_auc={float(score_dict['roc_auc']):.4f}\n")

# Calculate feature importances
complete_samples_FI_dict, complete_samples_encode_FI_df, complete_samples_raw_FI_df = pytorch_models.calculate_feature_importance(complete_data=this_all_dataset_variants['complete_samples'],method='random_forest',random_state=42)
utils.write_dict_to_pkl(complete_samples_FI_dict,f'{this_trial_path}/{this_model_name}/complete_samples_FI_dict.pkl')
complete_samples_encode_FI_df.to_csv(f'{this_trial_path}/{this_model_name}/complete_samples_encode_FI_df.csv',index=False)
complete_samples_raw_FI_df.to_csv(f'{this_trial_path}/{this_model_name}/complete_samples_raw_FI_df.csv',index=False)

normalise_missing_FI_scores = pytorch_models.calculate_missing_FI_scores(incomplete_trnX_missing_indicators=this_all_dataset_variants['incomplete_with_indicators']['trnX_missing_indicators'],
                                               raw_feature_names=this_raw_train_features,
                                               raw_feature_importance_dict=complete_samples_FI_dict['raw_normalise_importance_dict'])
print(normalise_missing_FI_scores)
# ---- SAVE ----
utils.write_dict_to_pkl(this_benchmark_performance_dict, f"{this_model_path}/{this_file_suffix}_benchmark_performance_dict.pkl")

#### Expected and Variance SV-based Utility Calculation

In [ ]:
import sv_utility_calculation
complete_trnX = this_all_dataset_variants['complete_samples']['trnX']
complete_trnY = this_all_dataset_variants['complete_samples']['trnY']
complete_trnX_idx = this_all_dataset_variants['complete_samples']['trnX_idx']
complete_devX = this_all_dataset_variants['complete_samples']['devX']
complete_devY = this_all_dataset_variants['complete_samples']['devY']

sv_algorithm = 'gradient_sv'
sv_optimiser = 'sgd'
sv_lr = 0.01
convergence_rate = 1e-3
sv_iterations = 100
sv_init_coreset_ratio = 1e-3
sv_cs_method = 'kmeans'
sv_total_cs_prop = 0.7
print(f'Init pts for SV: {int(sv_init_coreset_ratio*len(complete_trnX_idx))}, total Q pts: {int(len(complete_trnX_idx)*sv_total_cs_prop)}')
this_total_count = 1

sv_ml_task_config = {
    'model_name': 'RESNET',
    'complete_trnX': complete_trnX,
    'complete_trnX_idx': complete_trnX_idx,  
    'complete_trnY': complete_trnY,
    'complete_devX': complete_devX,
    'complete_devY': complete_devY,  
    'epochs': 100,
    'device': 'cpu',
    'metrics': 'both',
    'hidden_layer_list': [256, 128, 64],
    'auto_train_model': True,
}
sv_calculator = sv_utility_calculation.SVCalculator(sv_ml_task_config)
sv_alg_config = {
    'sv_algorithm': sv_algorithm,
    'max_iterations': sv_iterations,
    'optimiser':sv_optimiser,
    'lr': sv_lr,
    'convergence_rate': convergence_rate,
    'init_coreset_ratio': sv_init_coreset_ratio,
    'total_coreset_ratio': sv_total_cs_prop,
    'coreset_select_method': sv_cs_method,
}
sv_calculator.update_sv_alg_config(sv_alg_config)
utils.write_dict_to_pkl(sv_alg_config, f'{this_model_path}/sv_config_dict.pkl')
print(f'SV configuration: {sv_alg_config}')
# Calculate ground truth SV
sv_calculation_result_dict = sv_calculator.sv_calculation()
coreset_sv = sv_calculation_result_dict['coreset_sv']
utils.write_dict_to_pkl(sv_calculation_result_dict, f'{this_model_path}/sv_calculation_result_dict.pkl')
# Train surrogate model
sv_complete_pred_regressor = 'RF'
sv_predictor_config = {
    'complete_sv_pred_regressor': sv_complete_pred_regressor,
}
sv_predictor = sv_utility_calculation.SVModel(sv_ml_task_config)
sv_predictor.update_sv_predictor_config(sv_predictor_config)
surrogate_result_dict, sv_pred_exec_time = sv_predictor.sv_fit(sv_calculation_result_dict)
utils.write_dict_to_pkl(surrogate_result_dict, f'{this_model_path}/{sv_complete_pred_regressor}_surrogate_result_dict.pkl')

surrogate_result_dict = utils.read_dict_from_pkl(f'{this_trial_path}/{this_model_name}/RF_surrogate_result_dict.pkl')
complete_samples_preprocessor = this_all_dataset_variants['complete_samples']['preprocessor']
complete_samples_preprocessor_metadata = this_all_dataset_variants['complete_samples']['preprocessor_metadata']
complete_samples_marginals = this_all_dataset_variants['per_feature_marginals']
sv_predictor = sv_utility_calculation.SVPredict(
    surrogate_result_dict=surrogate_result_dict,
    preprocessor=complete_samples_preprocessor,
    preprocessor_metadata=complete_samples_preprocessor_metadata,
    marginals=complete_samples_marginals,
    numerical_cols=numerical_columns,
    categorical_cols=categorical_columns,
    device='cpu'  # Optional
)
incomplete_X_df = this_all_dataset_variants['incomplete_with_indicators']['trnX']
incomplete_sv_pred_results_dict = sv_predictor.predict_sv_incomplete(incomplete_X_df,n_imputations=100)
utils.write_dict_to_pkl(incomplete_sv_pred_results_dict, f'{this_trial_path}/{this_model_name}/incomplete_sv_pred_results_dict.pkl')
print('MEAN SV')
print(incomplete_sv_pred_results_dict['mean_sv'][:10])
print('STD SV')
print(incomplete_sv_pred_results_dict['std_sv'][:10])

#### usi data selection, imputation and downstream model training

In [ ]:

this_w_sv_list = [1.0]
this_w_unc_list = [0.05, 0.1]
this_w_mfi_list = [1.0, 0.5]
this_sampled_prop_list = [0.9]

sv_select_setting_combo_dict = {
    "ground_truth": {
        "incomplete_prop": this_sampled_prop_list,
        "descending": [False],
        "absolute": [True],
        "w_sv": this_w_sv_list,
        "w_unc": this_w_unc_list,
        "w_mfi": this_w_mfi_list,
    },
    "imputed": {
        "incomplete_prop": this_sampled_prop_list,
        "descending": [True],
        "absolute": [False],
        "w_sv": this_w_sv_list,
        "w_unc": this_w_unc_list,
        "w_mfi": this_w_mfi_list,
    },
}

all_sv_select_performance_dict = {"imputed": {}}

from itertools import product

# ---- controls ----
stop_metric = "roc_auc"

# max number of TRAINING restarts per (plugin, weight triple).
max_iteration = 10

# preprocess retry count (only used when preprocess throws a "known_ok" error)
max_preprocess_retry = 5

hyper_all_plugins = ['sklearn_missforest','miwae','sinkhorn','gain','hyperimpute','miracle','softimpute','most_frequent']
statistical_all_plugins = ['mice','mean']

all_plugins = hyper_all_plugins + statistical_all_plugins

print(f"Total plugins ({len(all_plugins)}): {all_plugins}")

total_no_of_incomplete_points = this_all_dataset_variants["incomplete_with_indicators"]["trnX"].shape[0]

# only imputed (as in your current setup)
tmp_sv_select_type = "imputed"
tmp_sv_select_setting_combo = sv_select_setting_combo_dict[tmp_sv_select_type]

# ---- loop: prop -> plugin -> weight-grid ----
for this_incomplete_selection_prop in tmp_sv_select_setting_combo["incomplete_prop"]:
    print("\n" + "=" * 90)
    print(
        f"Prop={this_incomplete_selection_prop} | "
        f"Select {int(this_incomplete_selection_prop * total_no_of_incomplete_points)} / {total_no_of_incomplete_points}"
    )

    # usi inputs
    incomplete_with_indicators = this_all_dataset_variants["incomplete_with_indicators"]
    incomplete_dataset_config = {
        "incomplete_trnX": incomplete_with_indicators["trnX"],
        "incomplete_trnY": incomplete_with_indicators["trnY"],
        "incomplete_trnX_idx": incomplete_with_indicators["trnX_idx"],
        "total_incomplete_select_ratio": this_incomplete_selection_prop,
        "normalised_missing_FI_scores": normalise_missing_FI_scores,
    }
    data_sampler = usi_data_selection.USI_sampler(incomplete_dataset_config)

    for plugin in all_plugins:
        print(f"\nPlugin={plugin}")

        best_score = -np.inf
        best_config_key = None
        best_score_dict = None
        best_iters_used = 0

        combo_iter = product(
            tmp_sv_select_setting_combo["w_sv"],
            tmp_sv_select_setting_combo["w_unc"],
            tmp_sv_select_setting_combo["w_mfi"],
        )

        for iter_i, (tmp_w_sv, tmp_w_unc, tmp_w_mfi) in enumerate(combo_iter, start=1):
            print(f"Processing plugin={plugin} weight_config_iter={iter_i}...")

            # seed used for SV-selection (keep selection reproducible per weight triple)
            seed = 12345 + 1000 * all_plugins.index(plugin) + iter_i
            utils.seed_everything(seed)

            config_key = (
                f"des_True_abs_False_"
                f"wsv_{tmp_w_sv}_wunc_{tmp_w_unc}_wmfi_{tmp_w_mfi}"
            )

            sv_data_sampler_config = {
                "descending": True,
                "absolute": False,
                "w_sv": tmp_w_sv,
                "w_unc": tmp_w_unc,
                "w_mfi": tmp_w_mfi,
            }

            # SV-select incomplete indices
            _, _, tmp_sv_incomplete_pos_idx_dict = data_sampler.sv_incomplete_sampler(
                sv_data_sampler_config,
                incomplete_sv_pred_results_dict,
            )
            sv_selected_incomplete_indices = list(
                tmp_sv_incomplete_pos_idx_dict["selected_incomplete_trnX_idx"]
            )

            # complete samplese
            selected_complete_indices = list(this_all_dataset_variants["complete_samples"]["trnX_idx"])

            # build training df:
            # - selected incomplete rows from simulate_train_df
            # - complete rows from og_train
            simulte_incomplete_train_df = simulate_train_df.loc[sv_selected_incomplete_indices]
            complete_train_df = og_train.loc[selected_complete_indices]
            tmp_sv_select_train_df = pd.concat([simulte_incomplete_train_df, complete_train_df])

            # choose imputation method
            if plugin in hyper_all_plugins:
                this_imputation_methods = f"hyper_{plugin}"
            elif plugin in statistical_all_plugins:
                this_imputation_methods = {"numerical": f"{plugin}", "categorical": "__MISSING__"}
            else:
                raise ValueError(f"Unknown plugin group: {plugin}")

            # ----------------------------
            # 1) preprocess with retry (only if "known_ok" error)
            # ----------------------------
            prep = None
            last_preprocess_error = None

            for retry_i in range(1, max_preprocess_retry + 1):
                preprocess_seed = seed + 10_000 * retry_i  # different from selection seed

                sv_select_data_preprocessor_config = {
                    "train_df": tmp_sv_select_train_df,
                    "val_df": og_validation,
                    "dev_df": simulate_develop_df,
                    "test_df": og_test,
                    "target_col": target_name,
                    "categorical_cols": categorical_columns,
                    "numerical_cols": numerical_columns,
                    "imputation_method": this_imputation_methods,
                    "create_variant": False,
                    "random_state": preprocess_seed,
                }

                try:
                    with torch.enable_grad():
                        prep = data_preprocessor.DataPreprocessor(sv_select_data_preprocessor_config).preprocess()
                    last_preprocess_error = None
                    break
                except (ValueError, RuntimeError, FloatingPointError, MemoryError) as e:
                    msg = str(e)
                    last_preprocess_error = e

                    known_ok = any(s in msg for s in [
                        "Expected parameter scale",   # MIWAE
                        "nan", "NaN", "inf", "Inf",
                        "singular", "Singular",
                        "cholesky", "Cholesky",
                        "CUDA out of memory",
                        "does not require grad"
                    ])

                    if known_ok:
                        print(
                            f"[preprocess retry] plugin={plugin} weight_config_iter={iter_i} "
                            f"retry={retry_i}/{max_preprocess_retry} seed={preprocess_seed}. Error: {e}"
                        )
                        continue

                    raise  # unknown error -> don't hide it

            if prep is None:
                print(
                    f"[preprocess failed] plugin={plugin} weight_config_iter={iter_i} "
                    f"config={config_key}. Last error: {last_preprocess_error}"
                )
                continue

            # ----------------------------
            # 2) Find the optimal w_unc and w_mfi
            # ----------------------------
            best_score_this_config = -np.inf
            best_score_dict_this_config = None
            best_train_restart_used = 0

            for train_restart_i in range(1, max_iteration + 1):
                train_seed = seed + 100_000 + train_restart_i
                utils.seed_everything(train_seed)

                new_sv_select_model = {
                    "model_name": this_model_name,
                    "trnX": prep["trnX"],
                    "trnY": prep["trnY"],
                    "tstX": prep["tstX"],
                    "tstY": prep["tstY"],
                    "valX": prep["valX"],
                    "valY": prep["valY"],
                    "epochs": 100,
                    "eval_metrics": "both",
                    "hidden_layer_list": [256, 128, 64],
                    "set_scheduler": True,
                }

                try:
                    with torch.enable_grad():
                        score_dict, pred_proba, model_params, _ = ModelEvaluator(
                            new_sv_select_model
                        ).evaluate_avg_metrics(total_count=1)
                except (ValueError, RuntimeError, FloatingPointError, MemoryError) as e:
                    msg = str(e)
                    known_ok = any(s in msg for s in [
                        "Expected parameter scale",
                        "nan", "NaN", "inf", "Inf",
                        "singular", "Singular",
                        "cholesky", "Cholesky",
                        "CUDA out of memory",
                    ])
                    if known_ok:
                        print(
                            f"[train retry] plugin={plugin} weight_config_iter={iter_i} "
                            f"train_restart={train_restart_i}/{max_iteration} seed={train_seed}. Error: {e}"
                        )
                        continue
                    raise

                this_roc = float(score_dict[stop_metric])

                if this_roc > best_score_this_config:
                    best_score_this_config = this_roc
                    best_score_dict_this_config = score_dict
                    best_train_restart_used = train_restart_i

            # update best across weight triples
            if best_score_this_config > best_score:
                best_score = best_score_this_config
                best_config_key = config_key
                best_score_dict = best_score_dict_this_config
                best_iters_used = iter_i

        best_stop_reason = "searched_all_weight_triples_and_restarts"

        # ---- store in your requested structure ----
        all_sv_select_performance_dict["imputed"].setdefault(plugin, {})
        all_sv_select_performance_dict["imputed"][plugin].setdefault(this_incomplete_selection_prop, {})
        all_sv_select_performance_dict["imputed"][plugin][this_incomplete_selection_prop]["config_key"] = best_config_key
        all_sv_select_performance_dict["imputed"][plugin][this_incomplete_selection_prop]["performance"] = best_score_dict
        all_sv_select_performance_dict["imputed"][plugin][this_incomplete_selection_prop]["meta"] = {
            "best_score": best_score,
            "iters_used": best_iters_used,   # weight_config_iter that achieved best
            "stop_reason": best_stop_reason,
        }

        print(f"    Best {stop_metric}={best_score:.6f} @ {best_config_key} | {best_stop_reason}")

        utils.write_dict_to_pkl(
            all_sv_select_performance_dict,
            f"{this_model_path}/{this_file_suffix}_select_{str(this_sampled_prop_list)}_sv_performance_dict.pkl",
        )

print("\n--------------------- ALL DONE -------------------")